In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.ticker import MaxNLocator
from matplotlib import gridspec
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
#plt.style.use('seaborn')
import seaborn as sns
import os

import glob
import datetime
import numpy as np
import pandas as pd
import math
from random import randrange
from random import randint
from tqdm import tqdm
import pickle
import re
import json



In [ ]:
def  savedicti (dictio, name):
    with open("/Users/friedrichjohenning/Desktop/dictionariesneurocorr/"+name+".txt", 'wb') as dict_items_save:
        pickle.dump(dictio, dict_items_save)
def  loaddicti (name):
    fileo= open("/Users/friedrichjohenning/Desktop/dictionariesneurocorr/"+name+".pickle", 'rb')
    #fileo= open("/Users/friedrichjohenning/Desktop/dictionaries/"+name+".pickle", 'rb')
    dictio=pickle.load(fileo)
            
    return dictio  




def checkConsecutive(l,n=3):
    # n = len(l) - 1
    n = n-1 # diff will be one less than all elements in the list
    return (sum(np.diff(sorted(l)) == 1) >= n) 

#@title Plotting setting 
## Plotting setting
plt.rcParams["font.family"] = "Arial"
plt.rcParams.update({'font.size': 10})
my_color_map = ['#56b4e9',
                '#e69f00',
                '#009e73',
                '#f0e442',
                '#0072b2',
                '#d55e00',
                '#cc79a7']



In [ ]:
aurocslow=loaddicti("aurocslow")
aurocfast=loaddicti("aurocfast")
PSTH_trace_milk=loaddicti("PSTHslow")
PSTH_trace_milkbinge=loaddicti("PSTHfast")
PSTH_trace_empty=loaddicti("PSTHempty")
mouse=loaddicti("everysingleexp")
dictofdataframes=loaddicti("concexpdayspermouse")

#why are values in these dicts missing?

In [ ]:
## picking out the positive slow milk responders based on auroc
idx_positiveslow = []


for key,value in aurocslow.items():
    
    

    data_auc = value.copy()
      
    data_aunp=np.asarray(data_auc)
    #print (type(data_aunp))
    
    if len(data_aunp)<31:
        data_aunp=np.zeros(50)
    
    #print (len(data_aunp))
    threshold_high = data_aunp[0:9].mean() + data_aunp[0:9].std()*3.5
      
    j_temp = []
    data_temp = []
    #print(key)
    
    for j in range(10,49):  # Only look at first 2 sec after delivery
        if data_aunp[j] > threshold_high:
            j_temp.append(j)
        if j_temp !=[]:
            if checkConsecutive(j_temp,n=4) == True:
                
                idx_positiveslow.append(key)
          # print('# {} is responding to milk'.format(idx))
        
idx_positiveslow=list(set(idx_positiveslow))        
print ("number of slow milk responders:"+str(len(idx_positiveslow)))

In [ ]:
## picking out the positive fast milk responders based on auroc

idx_positivefast = []


for key,value in aurocfast.items():
    
    data_auc = value.copy()
      
    data_aunp=np.asarray(data_auc)
    #print (type(data_aunp))
    
    if len(data_aunp)<31:
        data_aunp=np.zeros(50)
    
    #print (len(data_aunp))
    threshold_high = data_aunp[0:9].mean() + data_aunp[0:9].std()*3.5
      
    j_temp = []
    data_temp = []
    #print(key)
    
    for j in range(10,49):  # Only look at first 2 sec after delivery
        if data_aunp[j] > threshold_high:
            j_temp.append(j)
        if j_temp !=[]:
            if checkConsecutive(j_temp,n=4) == True:
                
                idx_positivefast.append(key)
          # print('# {} is responding to milk'.format(idx))
idx_positivefast=list(set(idx_positivefast))
print ("number of fast milk responders:" + str(len(idx_positivefast)))

# Plotting

## auroc based

In [ ]:
#auroc based on specific slow responders

fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,4])

new_df = pd.DataFrame()
new_dffast = pd.DataFrame()

for key in aurocslow.keys():
    #print (key)
    #print (len(aurocslow[key]))
    if len(aurocslow[key])<50:
        
        aurocslow[key].extend(np.zeros(50-len(aurocslow[key])))
        
    if len(aurocslow[key])==0:
       
        aurocslow[key]=np.zeros(50)
           
    
    
    
    new_df[key]=np.asarray(aurocslow[key]) 
    
for key in aurocfast.keys():
    #print (key)
    #print (type(aurocslow[key]))
    if len(aurocfast[key])<50:
        
        aurocfast[key].extend(np.zeros(50-len(aurocfast[key])))
        
    if len(aurocfast[key])==0:
       
        aurocfast[key]=np.zeros(50)
           
    
    
    
    new_dffast[key]=np.asarray(aurocfast[key]) 
        
    
    

isortslow=new_df.reindex(columns=idx_positiveslow) 
isortfast=new_dffast.reindex(columns=idx_positiveslow) 

bsorted=(isortfast[10:48].mean(axis=0))*-1  # -new_df[0:8].mean(axis=0)


sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isortslow = isortslow.reindex(columns=sorter)

isortfast=isortfast.reindex(columns=sorter)
isortslow.index=np.arange(0,5,0.1)
isortfast.index=np.arange(0,5,0.1)


axes[0].imshow(isortslow.T,cmap='viridis',aspect='auto',vmin=0,vmax=1)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')

axes[1].imshow(isortfast.T,cmap='viridis',aspect='auto',vmin=0,vmax=1)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/overviewPTSHauroc.pdf')

In [ ]:
#∆F/F of all  auroc slow response positive neurons and binge responses

fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,4])

new_df = pd.DataFrame()
new_dfbinge = pd.DataFrame()
new_dfauc = pd.DataFrame()
new_dfbingeauc =  pd.DataFrame()
for key in PSTH_trace_milk.keys():
    
           
    new_df[key]=(PSTH_trace_milk[key].mean(axis=1)) 
    # get average traces from each neurons/keys
    #for key in PSTH_trace_milkbinge.keys():
    #print(key)
    new_dfbinge[key]=(PSTH_trace_milkbinge[key].mean(axis=1))
    
    
new_dfauc = new_df.reindex(columns=idx_positiveslow)
new_dfbingeauc =  new_dfbinge.reindex(columns=idx_positiveslow)
bsorted=(new_dfbingeauc[1:4.7].mean(axis=0))*-1  #-new_dfbingeauc[0:8].mean(axis=0)



sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isortslow = new_dfauc.reindex(columns=sorter) # sort neuron index with responses 0 to +2 sec upon milk delivery
isortbinge=new_dfbingeauc.reindex(columns=sorter)
#isortshort=isort.loc[:,isort.columns[(isort.isnull().sum()!=50)&(isortbinge.isnull().sum() <48)]]
#isortbingeshort=isortbinge.loc[:,isort.columns[(isort.isnull().sum()!=50)&(isortbinge.isnull().sum()<48)]]



axes[0].imshow(isortslow.T,cmap='viridis',aspect='auto',vmin=-0.01,vmax=0.09)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')
axes[1].imshow(isortbinge.T,cmap='viridis',aspect='auto',vmin=-0.01,vmax=0.09)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/overviewPTSHdeltafauroc.pdf')

In [ ]:
#∆F/F of all  auroc slow response positive neurons and emptylicks

fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,4])

new_df = pd.DataFrame()
new_dfbinge = pd.DataFrame()
new_dfauc = pd.DataFrame()
new_dfempty =  pd.DataFrame()
for key in PSTH_trace_milk.keys():
    
           
    new_df[key]=(PSTH_trace_milk[key].mean(axis=1)) 
    # get average traces from each neurons/keys
    #for key in PSTH_trace_milkbinge.keys():
    #print(key)
    new_dfempty[key]=(PSTH_trace_empty[key].mean(axis=1))
    
    
new_dfauc = new_df.reindex(columns=idx_positiveslow)
new_dfemptyauc =  new_dfempty.reindex(columns=idx_positiveslow)
bsorted=(new_dfauc[1:4.7].mean(axis=0))*-1  #-new_dfbingeauc[0:8].mean(axis=0)



sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isortslow = new_dfauc.reindex(columns=sorter) # sort neuron index with responses 0 to +2 sec upon milk delivery
isortbinge=new_dfemptyauc.reindex(columns=sorter)
#isortshort=isort.loc[:,isort.columns[(isort.isnull().sum()!=50)&(isortbinge.isnull().sum() <48)]]
#isortbingeshort=isortbinge.loc[:,isort.columns[(isort.isnull().sum()!=50)&(isortbinge.isnull().sum()<48)]]



axes[0].imshow(isortslow.T,cmap='viridis',aspect='auto',vmin=-0.01,vmax=0.04)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')
axes[1].imshow(isortbinge.T,cmap='viridis',aspect='auto',vmin=-0.01,vmax=0.04)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/overviewPTSHdeltafaurocempty.pdf')

## traces and behavior

In [ ]:
#generates and plots behavior and all calcium trces of individual sweeps, not concatenated.
pump_num=[]  

for key,value in mouse.items():
    mouseId=key
    
    for key,value in value.items():
        expDate=key
        #print (key)
        for key,value in value.items():
            expId=key 
            if value[0].shape[0]>=1:
                pump_num.append(value[0].shape[0]-1)
            else:
                pump_num.append(value[0].shape[0])




ypsilon=np.vstack(np.arange(0,119.9,0.1))


for key,value in mouse.items():
    mouseId=key
    for keya,valuea in value.items():
        expDate=keya
        for keyb,valueb in valuea.items():
            expId=keyb 
            if valueb[0].shape[0]>=1:
                pump_num.append(valueb[0].shape[0]-1)
            else:
                pump_num.append(valueb[0].shape[0])
        for keyc,valuec in valuea.items():
            expId=keyc
            #print (valuec[0].shape)
            if valuec[4][0].shape[0]==0 or valuec[4][0].shape[1]==0:
                print('continue')
                continue
                
                
            #print (valuec[4][0])
            fig, ax = plt.subplots(figsize=[10,4],ncols=2,nrows=3,gridspec_kw={'width_ratios':[4,1]})
            ax[0,0].set_title(mouseId+" "+expDate+" "+expId)
            ax[0,0].eventplot(valuec[0],linelengths = 0.8,linewidths=0.6,lineoffsets = 1,color=my_color_map[0])
            ax[0,0].text(s=expDate,x=275,y=1,va='center',ha='right')
            sns.despine(left=True)
            ax[0,0].set_yticks([])
            ax[0,0].set_xlim([0,120])
            ax[1,0].eventplot(valuec[2],linelengths = 0.8,linewidths=0.6,lineoffsets = 1,color=my_color_map[1])
            ax[1,0].text(s=expDate,x=275,y=1,va='center',ha='right')
            sns.despine(left=True)
            ax[1,0].set_yticks([])
            ax[1,0].set_xlim([0,120])
            
            ax[2,0].plot(valuec[4][1],valuec[4][0].T,linewidth=0.2)
            ax[2,0].set_xlim([0,120])
            ax[2,0].set_ylim([-0.05,0.1])
            

In [ ]:
#plots behavior and responses of all cells
mouse_id="SNA095265"
for key, values in dictofdataframes[mouse_id].items():
    #print (mouse_Id)
    #print (key)
    #print (values[3].shape[1])
    zellen=values[3].shape[1]
    fig, ax = plt.subplots(figsize=[14,28],ncols=1,nrows=2+zellen,sharex=True)
    ax[0].set_title(key)
    ax[0].eventplot(values[0],linelengths = 0.2,linewidths=0.1,lineoffsets = 6,color=my_color_map[0])
    ax[0].eventplot(values[4],linelengths = 0.2,linewidths=0.1,lineoffsets = 6,color=my_color_map[5])
    #ax[0].text(s=file_date,x=275,y=idx,va='center',ha='right')
    sns.despine(left=True)
    ax[0].set_yticks([])
    #ax[0,0].set_xlim([0,120])
    ax[1].eventplot(values[1],linelengths = 0.2,linewidths=0.1,lineoffsets = 6,color=my_color_map[1])
    #ax[1].text(s=file_date,x=275,y=idx,va='center',ha='right')
    sns.despine(left=True)
    ax[1].set_yticks([])
    #ax[1,0].set_xlim([0,120])
    for n in range(zellen):
        ax[2+n].plot(values[2],values[3][n],linewidth=0.2)
        
        #ax[2+n].set_ylim([-0.05,0.1])

In [ ]:
#plot of example cell from labseminar
dataFrames=dictofdataframes["SNA095270"]
fig, ax = plt.subplots(figsize=[14,4],ncols=1,nrows=3,sharex=True)
#ax[0].set_title(key+str(n))
ax[0].eventplot(dataFrames["211123"][0],linelengths = 0.2,linewidths=0.5,lineoffsets = 6,color=my_color_map[0])
#ax[0].text(s=file_date,x=275,y=idx,va='center',ha='right')
sns.despine(left=True)
ax[0].set_yticks([])
#ax[0,0].set_xlim([0,120])
ax[1].eventplot(dataFrames["211123"][1],linelengths = 0.2,linewidths=0.1,lineoffsets = 6,color=my_color_map[1])
#ax[1].text(s=file_date,x=275,y=idx,va='center',ha='right')
sns.despine(left=True)
ax[1].set_yticks([])
ax[2].plot(dataFrames["211123"][2],dataFrames["211123"][3][8],linewidth=0.2)

fig.savefig('/Users/friedrichjohenning/Desktop/figurehung/211123570roi9.pdf')

In [ ]:
#PSTHs of all deltaF/F values
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=[7,16])

new_df = pd.DataFrame()
new_dfbinge = pd.DataFrame()
for key in PSTH_trace_milk.keys():
    print(key)
    new_df[key]=(PSTH_trace_milk[key].mean(axis=1)) 
    # get average traces from each neurons/keys
#for key in PSTH_trace_milkbinge.keys():
    #print(key)
    new_dfbinge[key]=(PSTH_trace_milkbinge[key].mean(axis=1))
bsorted=(new_dfbinge[9:48].mean(axis=0))*-1
#bsorted=(new_df[9:48].mean(axis=0)-new_df[0:8].mean(axis=0))*-1


sorted=bsorted.sort_values()

sorter=list(sorted.index)
   
isort = new_df.reindex(columns=sorter) # sort neuron index with responses 0 to +2 sec upon milk delivery
isortbinge=new_dfbinge.reindex(columns=sorter)
axes[0].imshow(isort.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[0].axvline(x=10,ls=':',color='k')
axes[0].set_title('time from pump activation')
axes[0].set_ylabel('neurons')
axes[0].set_xlabel('time bins (100 ms)')
axes[1].imshow(isortbinge.T,cmap='viridis',aspect='auto',vmin=-0.001,vmax=0.03)
axes[1].axvline(x=10,ls=':',color='k')
axes[1].set_title('time from pump activation')
axes[1].set_ylabel('neurons')
axes[1].set_xlabel('time bins (100 ms)')
